In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# === LOAD CORRECTLY (SPACE SEPARATED) ===
df = pd.read_csv('../datasets/german_credit.csv', sep='\s+', header=None)

print("Dataset 3 shape:", df.shape)

# Last column is the target in this dataset
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Convert labels {1,2} → {0,1}
y = y.replace({1: 1, 2: 0})

# Scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X))

print("Cleaned:", X_scaled.shape)

# === TRAIN MODELS ===
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    # probabilities for AUC
    proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'F1 Score': round(f1_score(y_test, pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, proba), 4)
    }

results_df = pd.DataFrame(results).T

print("\nDataset 3 Results:")
print(results_df)

results_df.to_csv('../results/german_credit_results.csv')
print("Dataset 3 done!")


Dataset 3 shape: (1000, 25)
Cleaned: (1000, 24)

Dataset 3 Results:
                     Accuracy  F1 Score  ROC-AUC
Logistic Regression     0.725    0.8057   0.7493
Random Forest           0.765    0.8418   0.8044
XGBoost                 0.740    0.8182   0.7594
Dataset 3 done!
